# How to search the full text of SEC filings with an API

Send a phrase, get back every 10-K, 10-Q, 8-K or 13D that contains it, with the matching passage, ticker and filing link. One POST request, with curl, Python and JavaScript.

Read it with charts at https://www.quantisnow.com/sec-api/guides/sec-full-text-search-api. More notebooks: https://github.com/quantisnow/sec-edgar-notebooks. Set `QUANTISNOW_API_KEY` before running; make a key at https://www.quantisnow.com/sec-api/keys.

Most SEC data APIs search what is about a filing: form type, filer, date. Full-text search looks at what the filing says. That is the only way to answer questions like which companies warned about their own survival this quarter, or which 8-Ks mention a cyber attack, because no form field records either.

## The request

`POST /v1/full-text` takes a `query` and, optionally, `formTypes`, `sort`, `size`, `from` and `facets`. A quoted phrase matches word for word, anywhere in the filing or its exhibits.

### Which companies said there is substantial doubt they can continue as a going concern?

Auditors and management have to say so when a company may not last another year. Credit desks and short sellers read this list every quarter.

```shell
curl -X POST https://www.quantisnow.com/api/v1/full-text \
  -H "Authorization: Bearer $QUANTISNOW_API_KEY" \
  -H "Content-Type: application/json" \
  -d '{
  "query": "\"substantial doubt\" AND \"going concern\"",
  "formTypes": ["10-Q", "10-Q/A"],
  "sort": "filedAt:desc",
  "size": 5,
  "facets": true
}'
```

Response (real, first 2 of 684):

```json
{
  "total": {
    "value": 684,
    "relation": "eq"
  },
  "took": 402,
  "data": [
    {
      "id": "0001493152-26-053834",
      "kind": "filing",
      "source": "SEC",
      "accessionNo": "0001493152-26-053834",
      "formType": "10-Q",
      "filedAt": "2026-11-23T17:27:40-05:00",
      "cik": "0001392694",
      "companyName": "SurgePays Inc.",
      "ticker": "SURG",
      "tickers": [
        "SURG"
      ],
      "items": [],
      "description": "SEC Form 10-Q filed by SurgePays Inc.",
      "linkToFilingIndex": "https://www.sec.gov/Archives/edgar/data/1392694/000149315226053834/0001493152-26-053834-index.htm",
      "quantisnowUrl": "https://www.quantisnow.com/insight/sec-form-10-q-filed-by-surgepays-inc-6804422",
      "score": null,
      "highlights": [
        "<em>doubt</em> about the Company's ability to continue as a <em>going concern</em> within one year after the date these consolidated financial statements are issued. 9 | Management has evaluated the significance of these conditions and has",
        "developed plans intended to mitigate the <em>substantial doubt</em>.",
        "Accordingly, management has concluded that its plans do not alleviate the <em>substantial doubt</em> about the Company's ability to continue as a <em>going concern</em>."
      ]
    },
    {
      "id": "0001477932-26-007251",
      "kind": "filing",
      "source": "SEC",
      "accessionNo": "0001477932-26-007251",
      "formType": "10-Q",
      "filedAt": "2026-11-23T17:27:21-05:00",
      "cik": "0001425627",
      "companyName": "SOBR Safe Inc.",
      "ticker": "SOBR",
      "tickers": [
        "SOBR"
      ],
      "items": [],
      "description": "SEC Form 10-Q filed by SOBR Safe Inc.",
      "linkToFilingIndex": "https://www.sec.gov/Archives/edgar/data/1425627/000147793226007251/0001477932-26-007251-index.htm",
      "quantisnowUrl": "https://www.quantisnow.com/insight/sec-form-10-q-filed-by-sobr-safe-inc-6804398",
      "score": null,
      "highlights": [
        "Accordingly, <em>substantial doubt</em> about the Company's ability to continue as a <em>going concern</em> has not been alleviated as of the date these financial statements are issued. 15 Table of Contents NOTE 3.",
        "Our current cash is not sufficient to fund our operations for the twelve months following the issuance of our financial statements, and there is <em>substantial doubt</em> about our ability to continue as a <em>going concern</em>.",
        "These conditions raise <em>substantial doubt</em> about the Company's ability to continue as a <em>going concern</em> for the twelve months following the date these financial statements are issued."
      ]
    }
  ],
  "facets": {
    "formType": [
      {
        "value": "10-Q",
        "count": 679
      },
      {
        "value": "10-Q/A",
        "count": 5
      }
    ],
    "month": [
      {
        "value": "2026-11",
        "count": 678
      },
      {
        "value": "2026-10",
        "count": 6
      }
    ],
    "companyName": [
      {
        "value": "Optimum Communications Inc.",
        "count": 3
      },
      {
        "value": "Apnimed Inc.",
        "count": 2
      },
      {
        "value": "Hepion Pharmaceuticals Inc.",
        "count": 2
      },
      {
        "value": "Spark I Acquisition Corp.",
        "count": 2
      },
      {
        "value": "1RT Acquisition Corp.",
        "count": 1
      },
      {
        "value": "22nd Century Group Inc.",
        "count": 1
      },
      {
        "value": "60 Degrees Pharmaceuticals Inc.",
        "count": 1
      },
      {
        "value": "8x8 Inc",
        "count": 1
      },
      {
        "value": "A SPAC III Acquisition Corp.",
        "count": 1
      },
      {
        "value": "AA Mission Acquisition Corp. II",
        "count": 1
      }
    ],
    "source": [
      {
        "value": "SEC",
        "count": 684
      }
    ]
  }
}
```

## Reading the response

Each record in `data` is one filing: `formType`, `filedAt` in US Eastern time, the filer's `cik`, `companyName` and `ticker`, its 8-K `items`, a `linkToFilingIndex` on sec.gov and up to three `highlights`. A highlight is a passage around the matches with each match in `<em>`, already HTML-escaped. `total.value` counts every match, not only the page.

In [ ]:
import os, re, requests

res = requests.post(
    "https://www.quantisnow.com/api/v1/full-text",
    headers={"Authorization": f"Bearer {os.environ['QUANTISNOW_API_KEY']}"},
    json={"query": '"substantial doubt" AND "going concern"', "formTypes": ["10-Q"], "size": 50},
)
for hit in res.json()["data"]:
    passage = re.sub(r"</?em>", "", hit["highlights"][0]) if hit["highlights"] else ""
    print(hit["ticker"], hit["filedAt"][:10], passage[:120])

## Narrowing with fields

Words and phrases search the text. A field narrows it: `formType:8-K`, `ticker:(AAPL OR MSFT)`, `items:"5.02"`, `filedAt:>=2026-11-01`. The 8-K item numbers are the most useful of these, because they say what kind of event the filing reports before you read a word of it.

### Which companies filed an 8-K to put a reverse stock split into effect?

Item 5.03 is the charter amendment that makes the split happen, so this finds the effective splits with their ratios, not proxies asking for permission.

```shell
curl -X POST https://www.quantisnow.com/api/v1/full-text \
  -H "Authorization: Bearer $QUANTISNOW_API_KEY" \
  -H "Content-Type: application/json" \
  -d '{
  "query": "\"reverse stock split\" AND items:\"5.03\"",
  "sort": "filedAt:desc",
  "size": 5
}'
```

Response (real, first 2 of 42):

```json
{
  "total": {
    "value": 42,
    "relation": "eq"
  },
  "took": 73,
  "data": [
    {
      "id": "0001683168-26-009069",
      "kind": "filing",
      "source": "SEC",
      "accessionNo": "0001683168-26-009069",
      "formType": "8-K",
      "filedAt": "2026-11-23T16:28:39-05:00",
      "cik": "0001554818",
      "companyName": "Auddia Inc.",
      "ticker": "AUUD",
      "tickers": [
        "AUUD"
      ],
      "items": [
        "3.03",
        "5.03",
        "9.01"
      ],
      "description": "Auddia Inc. filed SEC Form 8-K: Material Modification to Rights of Security Holders, Amendments to Articles of Incorporation or Bylaws; Change in Fiscal Year, Financial Statements and Exhibits",
      "linkToFilingIndex": "https://www.sec.gov/Archives/edgar/data/1554818/000168316826009069/0001683168-26-009069-index.htm",
      "quantisnowUrl": "https://www.quantisnow.com/insight/auddia-inc-filed-sec-form-8-k-material-modification-to-rights-6804142",
      "score": null,
      "highlights": [
        "The Certificate of Amendment provides that the <em>reverse stock split</em> became effective as of 5:00 P.M.",
        "The new CUSIP number for the common stock following the <em>reverse stock split</em> is 05072K503.",
        "In November 2026, the Company's board subsequently approved the final <em>reverse stock split</em> ratio of 1-for-13.9."
      ]
    },
    {
      "id": "0001213900-26-123183",
      "kind": "filing",
      "source": "SEC",
      "accessionNo": "0001213900-26-123183",
      "formType": "8-K",
      "filedAt": "2026-11-23T16:15:27-05:00",
      "cik": "0001358190",
      "companyName": "IT Tech Packaging Inc.",
      "ticker": "ITP",
      "tickers": [
        "ITP"
      ],
      "items": [
        "3.03",
        "5.03",
        "9.01"
      ],
      "description": "IT Tech Packaging Inc. filed SEC Form 8-K: Material Modification to Rights of Security Holders, Amendments to Articles of Incorporation or Bylaws; Change in Fiscal Year, Financial Statements and Exhibits",
      "linkToFilingIndex": "https://www.sec.gov/Archives/edgar/data/1358190/000121390026123183/0001213900-26-123183-index.htm",
      "quantisnowUrl": "https://www.quantisnow.com/insight/it-tech-packaging-inc-filed-sec-form-8-k-material-modification-6804081",
      "score": null,
      "highlights": [
        "The Company has obtained approval of the <em>Reverse Stock Split</em> by the NYSE American.",
        "The Company will issue one whole share of the post-<em>Reverse Stock Split</em> Common Stock to any stockholder who otherwise would have received a fractional share as a result of the <em>Reverse Stock Split</em>."
      ]
    }
  ]
}
```

## Cutting boilerplate with proximity

A phrase can be too strict and a pair of words too loose. Almost every 10-Q mentions tariffs and almost every one mentions gross margin; few say tariffs moved gross margin. `"tariffs gross margin"~12` asks for the words within 12 positions of each other, in any order, and is the most useful operator for getting past risk-factor language.

### Which quarterly reports tie tariffs to gross margin?

Nearly every 10-Q mentions tariffs somewhere. Asking for "tariffs" within 12 words of "gross margin" keeps the ones where tariffs moved the numbers.

```shell
curl -X POST https://www.quantisnow.com/api/v1/full-text \
  -H "Authorization: Bearer $QUANTISNOW_API_KEY" \
  -H "Content-Type: application/json" \
  -d '{
  "query": "\"tariffs gross margin\"~12",
  "formTypes": ["10-Q"],
  "size": 5,
  "facets": true
}'
```

Response (real, first 2 of 34):

```json
{
  "total": {
    "value": 34,
    "relation": "eq"
  },
  "took": 100,
  "data": [
    {
      "id": "0001437749-26-034949",
      "kind": "filing",
      "source": "SEC",
      "accessionNo": "0001437749-26-034949",
      "formType": "10-Q",
      "filedAt": "2026-11-04T16:16:05-05:00",
      "cik": "0001854139",
      "companyName": "Zevia PBC",
      "ticker": "ZVIA",
      "tickers": [
        "ZVIA"
      ],
      "items": [],
      "description": "SEC Form 10-Q filed by Zevia PBC",
      "linkToFilingIndex": "https://www.sec.gov/Archives/edgar/data/1854139/000143774926034949/0001437749-26-034949-index.htm",
      "quantisnowUrl": "https://www.quantisnow.com/insight/sec-form-10-q-filed-by-zevia-pbc-6772847",
      "score": 4.29,
      "highlights": [
        "<em>Gross margin</em> for the three months ended September 30, 2026 increased to 46.4% from 45.6% in the prior-year period.",
        "The increase in gross profit of $7.4 million, or 12.5%, was primarily due to higher volumes, the benefit of pricing actions, and lower spend on promotional activity, partially offset by unfavorable unit costs driven by <em>tariffs</em>",
        "<em>Gross margin</em> for the nine months ended September 30, 2026 decreased to 47.9% from 48.1% in the prior-year period."
      ]
    },
    {
      "id": "0001921963-26-000082",
      "kind": "filing",
      "source": "SEC",
      "accessionNo": "0001921963-26-000082",
      "formType": "10-Q",
      "filedAt": "2026-11-06T13:52:47-05:00",
      "cik": "0001921963",
      "companyName": "Atmus Filtration Technologies Inc.",
      "ticker": "ATMU",
      "tickers": [
        "ATMU"
      ],
      "items": [],
      "description": "SEC Form 10-Q filed by Atmus Filtration Technologies Inc.",
      "linkToFilingIndex": "https://www.sec.gov/Archives/edgar/data/1921963/000192196326000082/0001921963-26-000082-index.htm",
      "quantisnowUrl": "https://www.quantisnow.com/insight/sec-form-10-q-filed-by-atmus-filtration-technologies-inc-6778590",
      "score": 3.74,
      "highlights": [
        "The favorable impact from pricing is primarily driven by normal pricing initiatives and select adjustments as a result of <em>tariffs</em>.",
        "<em>Gross margin</em> Gross margin was $445.1 million for the nine months ended September 30, 2026 , an increase of $74.2 million compared to $370.9 million for the nine months ended September 30, 2025 ."
      ]
    }
  ],
  "facets": {
    "formType": [
      {
        "value": "10-Q",
        "count": 34
      }
    ],
    "month": [
      {
        "value": "2026-11",
        "count": 29
      },
      {
        "value": "2026-10",
        "count": 5
      }
    ],
    "companyName": [
      {
        "value": "Accuray Incorporated",
        "count": 1
      },
      {
        "value": "Alpha Pro Tech Ltd.",
        "count": 1
      },
      {
        "value": "Ambiq Micro Inc.",
        "count": 1
      },
      {
        "value": "Atmus Filtration Technologies Inc.",
        "count": 1
      },
      {
        "value": "BARK Inc.",
        "count": 1
      },
      {
        "value": "Bob's Discount Furniture Inc.",
        "count": 1
      },
      {
        "value": "Calix Inc",
        "count": 1
      },
      {
        "value": "CapsoVision Inc.",
        "count": 1
      },
      {
        "value": "Cerus Corporation",
        "count": 1
      },
      {
        "value": "Church & Dwight Company Inc.",
        "count": 1
      }
    ],
    "source": [
      {
        "value": "SEC",
        "count": 34
      }
    ]
  }
}
```

The [full-text search reference](https://www.quantisnow.com/sec-api/full-text-search) lists every parameter, operator and error, with a search box to try queries before you write code.